## Introduction 
Higher education has long been understood as a stepping stone to prepare young adults to enter the workforce. More specifically to teach and enhance skills necessary for the desired role of each student. However, a [Hult International Business School](https://www.hult.edu/blog/wi_skills_survey/) conducted a survey that reports students feel as though they are lacking skills when they enter the workforce. Only 24% of the recent graduates who were surveyed who worked in business roles said that they had all the skills needed for their current jobs. Over 77% graduates also say they learned more in 6 months of their jobs than in their entire undergraduate experience. Lastly, 96% of HR leaders believe that schools need to do a better job in workforce training. So, if students and employers feel dissatisfied with how undergraduate education is preparing people for future jobs, is there a skill gap between educational curriculums and skills required for jobs? 

Our study aims to hone into how Cornell courses are bridging, or failing to bridge, the gap between the skills taught at Cornell and the skills necessary for data jobs. Specifically, data scientists, data analysts, and data engineers in the year 2023. We are utilizing course roster data from the Cornell course roster API. Specifically filtering for skills mentioned in the course description. We are also using Luke Barousse’s public 2023 job-posting dataset for job required skills. The data sets will be described in more detail in the data description section of this report. 

### Research Questions 
1. How do the technical skills that employers request differ between US data analyst, data scientist and data engineer job postings in 2023?
2. How often are those skills mentioned in Cornell's computing and data related course descriptions from the same year (2023)?



The skills that are listed in job postings differed across the 3 roles. For example, 7.6% of the data engineer postings listed Docker compared with the 0.5% of data analyst postings. Then, when we compared the 20 most common skills in job postings the skills that were listed more often also tended to appear more often in Cornell course descriptions even though the rankings were different. Course descriptions may leave out skills taught in class which means that these results cant exactly tell us whether Cornell prepares students for these jobs.

## Data Description and Cleaning

The [data cleaning notebook](https://github.com/FrianaEngineer/INFO-2950-Group-Project/blob/main/phase3/phase3_cleaning.ipynb) explains how we cleaned the raw data and created the 2 CSV files that were used in this report: `job_postings_2023_clean.csv` and `cornell_courses_2023_clean.csv`.

### Job-posting data

We used [Luke Barousse’s data_jobs.csv dataset](https://huggingface.co/datasets/lukebarousse/data_jobs) which contains online job postings that are collected through Google from several job sites. Each row in our cleaned dataset represents one job posting. It contains 2 columns- job_title_short (which identifies the role as Data Analyst, Data Scientist or Data Engineer) and job_skills( which lists the skills extracted from that posting). We used the posting date and country from the original dataset in order to select the US postings from 2023 before creating this cleaned dataset. According to the dataset documentation the job titles were standardized using a BERT classification model and the skill lists were extracted from the postings using NLP (natural language processing).

We kept the postings from 2023 that were labeled United States and had a standardized job title of Data Analyst, Data Scientist or Data Engineer. We then removed exact duplicate rows and excluded postings without a usable skill list. If a job posting listed Python (or any other term) twice then we counted it as one posting that listed Python. After cleaning, we had 141,910 postings. For each role- we calculated the percentage mentioning a skill by dividing the number of postings that listed that skill by the total number of postings for that role with a usable skill list.

Barousse collected job listings through Google which means that the dataset may miss jobs advertised offline or listings that didn’t appear in his search results. The dataset may miss some skills listed in a posting or put a posting in the wrong job category.

[Hugging Face dataset and documentation](https://huggingface.co/datasets/lukebarousse/data_jobs) · [Original CSV file](https://huggingface.co/datasets/lukebarousse/data_jobs/tree/main)

### Cornell course data

We created this dataset to measure how often specific skills appear in Cornell roster descriptions. We used the [Cornell Class Roster API](https://classes.cornell.edu/content/FA25/api-details) to collect course descriptions from Spring and Fall 2023. We included four subjects related to data careers and included both undergraduate and graduate courses. These course categories cover computing, information science, statistics and quantitative modeling: computer science (CS), Information Science (INFO), Statistics and Data Science (STSCI), and Operations Research and Information Engineering (ORIE). We decided to select courses by their subject codes before checking which skills appeared in their descriptions.

We kept courses with descriptions that were not blank and counted each course ID once across both semesters (prevents courses offered in both semesters from receiving extra weight). If a course had 2 different descriptions then we searched both and counted the course once if either description mentioned the skill. After cleaning, we had 398 distinct courses. The cleaned dataset has two columns- crseId( identifies each course) and description (contains the course description text we searched for skills)
The reason why we used course IDs instead of matching titles and descriptions is because different courses can have the same title or description.

Cornell publishes the class roster in order to help students choose courses. The descriptions come directly from Cornell so they show how the university describes its courses. However, these descriptions summarize the courses and may leave out tools that may be used in assignments or lectures. Hence we use them to measure skill mentions- not everything that students learn.

The [raw Cornell data](https://github.com/FrianaEngineer/INFO-2950-Group-Project/tree/main/phase2/raw) are saved as JSON files in our repository we used the SP23 and FA23 files for CS, INFO, STSCI and ORIE.

## Revised Hypotheses and Analysis Plan

In Phase 2- our first analysis had 11 yearly observations and our second had 10 year-to-year changes. After we got feedback we decided to change our question to compare specific skills in 2023 job postings and Cornell course descriptions. We had already made an exploratory table of the 20 most common skills before writing this revised plan.

### Hypothesis 1

We expect the Data Engineer postings to list docker more often than Data Analyst postings. [Docker’s documentation](https://docs.docker.com/get-started/docker-overview/) explains that it packages applications into containers so that they can run consistently in different environments. This may be useful when it comes to deploying data pipelines- this is why we would expect Docker to appear more often in data engineer postings than data analyst postings.

**Planned analysis:** For each posting we will record if it lists docker. We will compare the percentages across the 3 job roles and use logistic regression to test if the odds of listing docker are higher for Data Engineer postings compared to Data Analyst postings.

### Hypothesis 2

We expect skills that appear more often in job postings to also appear more often in Cornell course descriptions. Tools that are useful for many kinds of data work may come up in both job postings and courses.

**Planned analysis:** For each of the 20 skills we will calculate the percent of job postings across all 3 roles that list the skill and the percent of Cornell courses whose descriptions mention it. We will use Spearman correlation to measure if skills that rank higher in job postings also tend to rank higher in Cornell course descriptions.

We will first calculate the correlation by pairing each skill’s job posting percent with its own Cornell percent. Then we will keep the job posting percents in place and randomly shuffle the Cornell percentages 10,000 times. we will count how many times shuffling gives us a correlation = to or greater than the correlation we got when we matched each skill to itself.

Before choosing this test, we had already looked at the job posting and Cornell percents for all 20 skills. We treat the test as exploratory because it measures a pattern in data that we had already seen.

Each observation in this analysis represents 1 skill which means that the correlation uses 20 observations. The job postings and courses are used to calculate the two percents for each skill.

In [1]:
# load the 2 cleaned datasets and turn the saved job skills into lists.
import ast
import pandas as pd

jobs = pd.read_csv("job_postings_2023_clean.csv")
courses = pd.read_csv("cornell_courses_2023_clean.csv")

# the csv stores each skill list as text. Convert it back to a list.
jobs["job_skills"] = jobs["job_skills"].map(ast.literal_eval)

print("Job postings:", len(jobs))
print("Cornell courses:", len(courses))

Job postings: 141910
Cornell courses: 398


### How we matched skills across the datasets

We selected the 20 skills that appeared in the most job postings across the three roles. We used the skill lists provided in the job dataset and searched the Cornell course descriptions for those same skills. A course counted once for a skill even if its description mentioned it several times.

In [2]:
# 20 most common skills and see how often each job role lists them.
postings_per_role = jobs["job_title_short"].value_counts()
job_skills = jobs.explode("job_skills", ignore_index=True)
# Select the 20 most common skills across all three roles.
top_skills = job_skills["job_skills"].value_counts().head(20).index

skill_counts = pd.crosstab(
    job_skills["job_skills"], job_skills["job_title_short"]
).reindex(top_skills)

skill_percentages = (
    skill_counts.div(postings_per_role, axis="columns") * 100
).round(1)

skill_percentages

job_title_short,Data Analyst,Data Engineer,Data Scientist
job_skills,,,
sql,60.0,71.9,58.1
python,32.0,68.2,82.0
r,20.3,9.5,50.4
tableau,33.7,15.1,26.9
excel,47.9,6.0,12.8
aws,5.8,45.0,19.9
spark,2.4,33.7,19.1
azure,5.0,33.8,13.3
power bi,20.2,10.4,11.0


In [3]:
# Count how many cornell course descriptions mention each of the 20 skills.
import re

# Use more specific patterns where a skill name can mean something else.
special_patterns = {
    "r": r"\bR\b",
    "excel": r"\bExcel\b",
    "word": r"(?i)\b(?:Microsoft|MS)\s+Word\b",
    "spark": r"(?i)\b(?:Apache\s+Spark|PySpark)\b|\bSpark\b",
    "azure": r"(?i)\bMicrosoft\s+Azure\b|\bAzure\b",
    "oracle": r"\bOracle\b",
    "snowflake": r"\bSnowflake\b",
"sql": r"(?i)\bSQL\b(?!\s+Server\b|[-–—]\s*like\b)",}

descriptions = courses["description"].fillna("")
cornell_counts = {}

for skill in top_skills:
    pattern = special_patterns.get(skill, rf"(?i)\b{re.escape(skill)}\b")
    cornell_counts[skill] = descriptions.str.contains(pattern).sum()

comparison = skill_percentages.copy()
comparison["Cornell course count"] = pd.Series(cornell_counts)
comparison["Cornell courses (%)"] = (
    comparison["Cornell course count"] / len(courses) * 100
).round(1)

comparison

job_title_short,Data Analyst,Data Engineer,Data Scientist,Cornell course count,Cornell courses (%)
job_skills,,,,,
sql,60.0,71.9,58.1,2,0.5
python,32.0,68.2,82.0,16,4.0
r,20.3,9.5,50.4,26,6.5
tableau,33.7,15.1,26.9,0,0.0
excel,47.9,6.0,12.8,3,0.8
aws,5.8,45.0,19.9,0,0.0
spark,2.4,33.7,19.1,1,0.3
azure,5.0,33.8,13.3,0,0.0
power bi,20.2,10.4,11.0,0,0.0


### Exploratory comparison of job postings and course descriptions

SQL appeared in at least 58% of postings with usable skill lists in each of the three roles. Other skills varied more by role. Excel appeared in 47.9% of data analyst postings, compared with 6.0% of data engineer postings and 12.8% of data scientist postings. Python appeared in 82.0% of data scientist postings, compared with 68.2% of data engineer postings and 32.0% of data analyst postings.

Among the 398 Cornell courses we examined, 26 descriptions mentioned R (6.5%), 16 mentioned Python (4.0%), and 2 mentioned SQL (0.5%). The two sources describe different things. Job postings list skills employers ask for, while short course descriptions may leave out tools students use in class. These results tell us which skills were explicitly mentioned in each source.

In [4]:
# Read the words around each R match to check that it means the language
for course_id, description in zip(courses["crseId"], descriptions):
    for match in re.finditer(special_patterns["r"], description):
        start = max(0, match.start() - 60)
        end = match.end() + 60
        snippet = description[start:end].replace("\n", " ")
        print(course_id, snippet)

352214 me securities, and resampling techniques. Covers the use of R for statistical calculations, simulation, and optimization.
352376 odeling and parallel computing. Programming will be done in R.
358529 ming (VBA), and programming in a scripting language (either R or Python). Operations research methods include inventory m
358808 les of statistical programming (such as data frames, Python/R packages, reproducible workflows, and version control), uni
359479  bayesian statistics. Students are instructed in the use of R software; prior knowledge of R is helpful but not required.
359479 are instructed in the use of R software; prior knowledge of R is helpful but not required. This course is intended for M.
360952  to linear modeling. Applied analysis is carried out in the R statistical computing environment.
364083 rs. Homework assignments will involve computation using the R statistical package.
364513 methods. Some assignments may involve computation using the R programming language.


In [5]:
# Show the words around Excel and Spark so we can check what they mean.
for skill in ["excel", "spark"]:
    pattern = special_patterns[skill]

    for course_id, description in zip(courses["crseId"], descriptions):
        for match in re.finditer(pattern, description):
            start = max(0, match.start() - 60)
            end = match.end() + 60
            snippet = description[start:end].replace("\n", " ")
            print(skill, course_id, snippet)

excel 358529 ery language (SQL), geographical information systems (GIS), Excel and Visual Basic programming (VBA), and programming in a sc
excel 359480 sion problems in a hands-on learning environment. Microsoft Excel is heavily used. A wide variety of application areas are co
excel 371823 sion problems in a hands-on learning environment. Microsoft Excel is heavily used. A wide variety of application areas are co
spark 367928 NumPy/Scipy, scikit-learn), parallel computing (Map-Reduce, Spark, Hadoop), basic finite-precision arithmetic, an overview of


### Checking the skill matches

We read the words around each R match in the Cornell course descriptions. All 37 matches referred to the R programming language and came from 26 distinct courses. Some descriptions mentioned R more than once. We also checked that the Excel matches referred to the spreadsheet software and that the Spark match referred to the tool used for parallel computing.

For job postings, we used the skill lists that came with the dataset. We spot-checked the Docker indicator by comparing it with the saved skill lists. The CSV does not include the original posting text, so this check only tells us whether our code counted the saved labels correctly. We also counted postings that listed Python, R or Excel directly from the saved skill lists. For each skill those counts matched our comparison table in all 3 job roles.

In [6]:
# mark each posting with a 1 if it lists Docker and a 0 if it doesnt.
jobs["lists_docker"] = [
    int("docker" in skills) for skills in jobs["job_skills"]
]

jobs[["job_title_short", "job_skills", "lists_docker"]].head()

,job_title_short,job_skills,lists_docker
0,Data Analyst,"[jira, python, sql]",0
1,Data Scientist,"[docker, python, r, splunk, tableau]",1
2,Data Scientist,"[alteryx, cognos, python, r, spss, sql, tableau]",0
3,Data Scientist,"[excel, python, r, sql, tableau]",0
4,Data Engineer,"[azure, bash, databricks, excel, git, hadoop, ...",0


In [7]:
# Find the % of postings in each job role that list Docker.
docker_by_role = jobs.groupby("job_title_short")["lists_docker"].agg(
    total_postings="size",
    postings_with_docker="sum"
)

docker_by_role["percent_with_docker"] = (
    100 * docker_by_role["postings_with_docker"]
    / docker_by_role["total_postings"]
).round(1)

docker_by_role

,total_postings,postings_with_docker,percent_with_docker
job_title_short,,,
Data Analyst,57411,305,0.5
Data Engineer,32814,2488,7.6
Data Scientist,51685,2791,5.4


In [8]:
# Compare the odds of listing Docker across job roles using logistic regression.
import statsmodels.formula.api as smf

# Predict whether a posting lists Docker using its job role.
# Use Data Analyst as the reference group.
docker_model = smf.logit(
    'lists_docker ~ C(job_title_short, Treatment(reference="Data Analyst"))',
    data=jobs
).fit(disp=False)

docker_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           lists_docker   No. Observations:               141910
Model:                          Logit   Df Residuals:                   141907
Method:                           MLE   Df Model:                            2
Date:                Tue, 10 Nov 2026   Pseudo R-squ.:                 0.08359
Time:                        13:55:26   Log-Likelihood:                -21571.
converged:                       True   LL-Null:                       -23539.
Covariance Type:            nonrobust   LLR p-value:                     0.000
=============================================================================================================================================
                                                                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                    -5.2324      0.057    -91.136      0.000      -5.345      -5.120
C(job_title_short, Treatment(reference="Data Analyst"))[T.Data Engineer]      2.7318      0.061     44.723      0.000       2.612       2.852
C(job_title_short, Treatment(reference="Data Analyst"))[T.Data Scientist]     2.3691      0.061     39.080      0.000       2.250       2.488
=============================================================================================================================================
"""

### Hypothesis 1 Results

We predicted that Data Engineer postings would list docker more often than Data Analyst postings. Out of the postings with usable skill lists 7.6% of Data Engineer postings listed Docker, compared with 0.5% of Data Analyst postings and 5.4% of Data Scientist postings.

We used job role to predict the probability that a posting listed Docker. Using Data Analyst as the reference group the model estimated that the odds of listing Docker were about 15.4 times as high for Data Engineer postings as for Data Analyst postings. Even in the role with the highest rate- fewer than 8 in 100 postings listed Docker.

The p value for the comparison between Data Engineer and Data Analyst postings was below 0.001. We rejected the null hypothesis that the two roles had equal odds of listing Docker. This supports our prediction for the 2023 postings with usable skill lists in our dataset.

*Data Scientist postings also had much higher odds of listing Docker than Data Analyst postings (about 10.7 times as high), so the main contrast is between analysts and the other two roles. Because job role is the only input, the model's predicted probabilities equal the group rates above (0.5% for analysts, 7.6% for engineers, and 5.4% for scientists). Role explains only a small part of which postings list Docker (pseudo R² = 0.084), since Docker is rare in every role.*

In [9]:
# job posting and Cornell course % side by side by skill.
skill_comparison = pd.DataFrame({
    "job_postings_pct": 100 * skill_counts.sum(axis=1)/len(jobs),
    "cornell_courses_pct": 100 * comparison["Cornell course count"]/len(courses)})

skill_comparison.head().round(1)

,job_postings_pct,cornell_courses_pct
job_skills,,
sql,62.1,0.5
python,58.6,4.0
r,28.8,6.5
tableau,26.9,0.0
excel,25.4,0.8


In [10]:
# Use spearman corr to see how closely the two skill rankings agree.
job_p = skill_comparison["job_postings_pct"]
cornell_p = skill_comparison["cornell_courses_pct"]

observed_correlation = job_p.corr(
    cornell_p, method="spearman"
)

print(f"Spearman correlation: {observed_correlation:.3f}")

Spearman correlation: 0.457


In [11]:
# shuffle Cornell % to understand our corr
import numpy as np
from scipy.stats import spearmanr

rng = np.random.default_rng(2950)
num_shuffles = 10_000
at_least_as_high = 0

for _ in range(num_shuffles):
    # Keep job %, shuffle the Cornell %.
    shuffled_cornell = rng.permutation(cornell_p)
    shuffled_correlation = spearmanr(job_p, shuffled_cornell)[0]

    if shuffled_correlation >= observed_correlation - 1e-12:
        at_least_as_high += 1

# Include the observed result in the count.
p_value = (at_least_as_high + 1) / (num_shuffles + 1)

print(f"at least as high as the obs corr: {at_least_as_high:,}")
print(f"One-sided p-value: {p_value:.4f}")

at least as high as the obs corr: 227
One-sided p-value: 0.0228


### Hypothesis 2 Results

We predicted that skills listed more often in job postings would also be mentioned more often in Cornell course descriptions. Across the 20 skills, the Spearman correlation was 0.457. This means skills that ranked higher in job postings tended to rank higher in Cornell course descriptions, although the rankings differed. For example, SQL ranked higher than R in job postings, while R ranked higher than SQL in Cornell course descriptions.

In 227 out of 10,000 random pairings, the correlation was at least as high as the one we observed. The one-sided p-value was 0.0228. Since this is below 0.05, we rejected the null hypothesis that the skill rankings were unrelated. This result supports our prediction for the 20 skills we studied. We treat it as exploratory because we had already looked at the skill comparison table before choosing this test.

### Additional checks (not preregistered)

Because the Hypothesis 2 correlation uses only 20 skills, we checked how much it depends on Python and R, the two skills that rank high in both job postings and Cornell course descriptions. We recomputed Spearman's ρ with each skill removed, and we computed a two-sided p-value for the full result.

In [12]:
# Extra checks on Hypothesis 2 (not preregistered)
rng_extra = np.random.default_rng(2950)

def perm_p(job, cornell, rho, n=10_000):
    one = two = 0
    for _ in range(n):
        r = spearmanr(job, rng_extra.permutation(cornell))[0]
        one += r >= rho - 1e-12
        two += abs(r) >= abs(rho) - 1e-12
    return (one + 1) / (n + 1), (two + 1) / (n + 1)

p_one, p_two = perm_p(job_p, cornell_p, observed_correlation)
print(f"All 20 skills: rho = {observed_correlation:.3f}, "
      f"one-sided p = {p_one:.4f}, two-sided p = {p_two:.4f}")

for drop in ["python", "r"]:
    keep = job_p.index != drop
    j, c = job_p[keep], cornell_p[keep]
    rho = j.corr(c, method="spearman")
    p_one, p_two = perm_p(j, c, rho)
    print(f"Without {drop}: rho = {rho:.3f}, "
          f"one-sided p = {p_one:.4f}, two-sided p = {p_two:.4f}")

All 20 skills: rho = 0.457, one-sided p = 0.0228, two-sided p = 0.0457
Without python: rho = 0.372, one-sided p = 0.0558, two-sided p = 0.1129
Without r: rho = 0.372, one-sided p = 0.0563, two-sided p = 0.1172


In both cases the correlation stays positive but is no longer significant at α = 0.05. The Hypothesis 2 result therefore depends on a few high-frequency skills, and we treat it as weak, exploratory evidence.

## Evaluation of significance

We used α = 0.05 for both hypotheses. The table summarizes each test, and the full analyses are in the Data analysis section.

| | Hypothesis 1 (Docker by role) | Hypothesis 2 (skills vs. courses) |
|---|---|---|
| Test | Logistic regression | Spearman correlation with a permutation test |
| Result | Odds ratio 15.4 (Data Engineer vs. Data Analyst) | ρ = 0.457 across 20 skills |
| p-value | < 0.001 (two-sided) | 0.0228 (one-sided); 0.0457 (two-sided) |
| α | 0.05 | 0.05 |
| Decision | Reject the null | Reject the null (weak evidence) |
| Effect size | 7.6% vs. 0.5% of postings list Docker | Moderate (ρ ≈ 0.46) |
| Main caveat | Postings may be duplicated across boards | Exploratory; depends on Python and R |

**Hypothesis 1.** The p-value was below our α of 0.05, so we reject the null that the odds of listing Docker are equal for Data Engineer and Data Analyst postings. With 141,910 postings, even tiny differences would be significant, so the effect size matters more: 7.6% of engineer postings list Docker versus 0.5% of analyst postings. We removed only exact duplicate rows, so the same job may appear on several boards. That would make the p-value too small, but we would not expect it to erase an effect this large. We tested only one skill, so this does not show that every skill differs by role. The finding suggests that one skills list would not serve all three roles.

**Hypothesis 2.** The one-sided permutation p-value was 0.0228, below 0.05, so we reject the null of no association. We treat this as weak evidence. The test is exploratory because we looked at the skill table first. The two-sided p-value is 0.0457, just under 0.05, and ten of the 20 skills have no Cornell mentions. Dropping Python or R lowers ρ to 0.372 and raises the one-sided p-value to about 0.056, above 0.05, so the result depends on those two skills (see the additional checks above). Because 19 skills give the test less power, this does not show the association disappears. Courses and postings both emphasize widely used tools like Python and R, while cloud and BI tools such as AWS and Tableau rarely appear in course descriptions.

## Conclusion
For this project we set out to see how the skills and material that we are learning in college matches the skills and knowledge that is required for our potential careers. Our goal was to see if there was a gap between the curriculums that are taught at Cornell and the skills that are actively sought out by employers. We used Python with pandas, NumPy, SciPy, and statsmodels for the analysis, and the requests library to collect the Cornell roster data. We decided to look at three main jobs that might interest students that are taking this class, mainly data analysts, data scientists, and data engineers. We then compared these terms to the course descriptions for Cornell classes from the spring and fall semesters of 2023 under the categories of CS, INFO, ORIE, and STSCI. 

Over the course of this project we analysed over 141,000 U.S. job postings for data scientists, analysts, and engineers and compared them to almost 400 Cornell courses in CS, INFO, ORIE, and STSCI. Our first analysis examined whether the required skills varied between the three different jobs that we were studying. We did this by looking at a specific skill called Docker which is a tool used to build data pipelines and software packaging. Using a logistic regression model, we were able to find that data engineers had significantly higher odds of needing to know how to use Docker according to their job description compared to data analysts. We found that data engineers' odds of needing Docker were actually 15.4 times higher than data analysts with a p-value < 0.001. This helped us to confirm that the skills that are required by job postings do differ between different roles. 

Our second analysis tested whether the amount of times that a skill was mentioned in the job descriptions was at all correlated with how often they appeared in the Cornell course descriptions. Although the percentage of courses specifically mentioning the skills that were seen in the job descriptions were relatively low we did still see a statistically significant Spearman rank correlation of 0.457 with p = 0.0228. This suggests a moderately positive alignment between how much skills are needed for data based jobs and how much these skills are seen in Cornell curriculums. However, this result depends on Python and R. Without either one, the correlation is no longer significant at α = 0.05, so we treat it as weak, exploratory evidence. Overall this likely reflects the difference in the tasks and goals of job descriptions and courses at Cornell. College courses are often focused on teaching broad skills that can be used for a wide variety of different purposes in order to cater to many different students whereas job postings are often focused on very narrow and specific skills that that specific role would require.


## Limitations

While the findings of our project offer a window into the relationship between industry skills and college curriculums there are several limitations that should be considered:

Course Descriptions: Our analysis of Cornell coursework relied entirely on official course descriptions from the Class Roster API, this is a limitation due to the fact that there are classes that may teach specific skills that are not mentioned in their course description. They could be missing details about in class activities or lectures that would help students to gain the skills that are needed for industry jobs. This could mean that we are underestimating the number of Cornell courses that are actually teaching these skills.

Job Posting Dataset: Luke Barousse’s dataset collects online job postings through Google search results, this could lead to a bias due to the fact that only online postings and information are collected. This does not consider job offers from in person events like career fairs and could possibly skip over job postings on sites that require logins to view postings. The other limitation for this data set is that it also is not sorted by experience level. Students that are coming right out of college often need a much different skill set than someone who is applying for a position much later in their career. 
Barousse collected these postings in order to help people explore the data job market and included the skills that employers request and the salaries they offer.

Impact: The limitations of our data and our study could lead to us under estimating the amount of skills that are covered by college courses. This is because we could be underestimating what classes are actually teaching as well as what jobs really require. This could lead to a conclusion that undervalues the education that we are receiving and its relevance to our future careers.




## Acknowledgements and Bibliography 
Brown, E. (n.d.). New survey reveals traditional undergraduate education is not preparing students for the workforce : Hult International Business School. New Survey Highlights Skills Gap in Recent College Graduates. https://www.hult.edu/blog/wi_skills_survey/  

Cornell University, O. of the U. R. (n.d.). Class roster. Class Roster - Fall 2025 - API Details. https://classes.cornell.edu/content/FA25/api-details 

Lukebarousse/data_jobs · Datasets at hugging face. lukebarousse/data_jobs · Datasets at Hugging Face. (n.d.). https://huggingface.co/datasets/lukebarousse/data_jobs 

Docker, Inc. (n.d.). *Docker overview*. https://docs.docker.com/get-started/docker-overview/

We used Python with pandas, NumPy, SciPy and statsmodels for our analysis. 

## Questions for Reviewers

1. We chose CS, INFO, STSCI, and ORIE before counting any skills. ORIE is in Engineering while the others are in CIS, and we left out Math and Econ. Is hand-picking four subjects defensible, or should we use a college-based rule such as CIS plus Engineering?
2. The job skill lists come from the dataset author's NLP extraction, and we can't check them against posting text. We checked our keyword matches in Cornell descriptions only for R, Excel, and Spark. How much validation is enough?
3. We removed only exact duplicate rows, but the same job can appear on several job boards. How should we deduplicate, and how should that change how we read the p-value?
4. Hypothesis 2 uses 20 skills, and half have no Cornell mentions. Is a rank correlation appropriate here, or should we include more skills or use another statistic?
5. We looked at the skill table before choosing our tests, and the result depends on Python and R. How strongly should we word our conclusions?